<img src="../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

<style>
h1 {
  font-size: 2.35rem !important;
  line-height: 1.15 !important;
  margin: 0.7rem 0 1rem !important;
}
h2 {
  font-size: 1.9rem !important;
  line-height: 1.2 !important;
  margin: 0.6rem 0 1rem !important;
}
h3 {
  font-size: 1.35rem !important;
  line-height: 1.3 !important;
  margin: 1.9rem 0 0.75rem !important;
}
h4 {
  font-size: 1.08rem !important;
  line-height: 1.35 !important;
  margin: 1.35rem 0 0.65rem !important;
}
.alchemi-api-label,
span[aria-label="ALCHEMI Toolkit API"] {
  color: currentColor !important;
  font-size: 0.70rem !important;
  font-weight: 700;
  letter-spacing: 0.05em;
}
@supports (color: color-mix(in srgb, black, white)) {
  .alchemi-api-label,
  span[aria-label="ALCHEMI Toolkit API"] {
    color: color-mix(in srgb, currentColor 68%, #76B900 32%) !important;
  }
}
</style>

# ALCHEMI Core · Module 03: Scale a study

**Goal:** Keep a two-stage MD study moving through one active `Batch`.

**Estimated time:** 15–20 minutes  
**Next step:** [Return to the course overview →](../../README.md)

<img src="assets/course-roadmap-module03.svg" alt="ALCHEMI Core roadmap with Run in stages highlighted in Module 03" style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:1rem 0 0;">

In [23]:

import torch
from helpers import module03 as helpers

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)
if device.type == "cuda":
    print("gpu:   ", torch.cuda.get_device_name(device))

device: cuda
gpu:    NVIDIA RTX 4000 SFF Ada Generation


Module 02 ran MD simulations on three selected complexes. Real screening studies can span hundreds, thousands, or even millions of systems. Here, we'll learn the ropes with a fast study of a few hundred structures.

We will start from all 205 compatible NCIA250 `AB` complexes. First choose how many independent trajectories to prepare for each complex.

In [25]:
batch_multiplier = 2  # two trajectories per complex = 410 total

`batch_multiplier` controls how many independent trajectories we prepare. Set it to `0.5` for 103 trajectories, `1` for one trajectory from each of the 205 complexes, or `2` for 410 trajectories. We use `2` so the study contains several groups and leaves waiting work for inflight batching.

### Watch work move through the active `Batch`

The next cell shows schematic NVT → NVE timing for the three routes. Watch when trajectories change stage and when waiting trajectories enter the active `Batch`.

In [28]:
helpers.show_scheduling_widget()

## Route 1 · Run in stages

Real simulation studies often use several MD stages, such as temperature-controlled NVT followed by energy-conserving NVE. This lesson uses that compact NVT → NVE path to teach scheduling.

The simplest route processes one group at a time. The whole group completes NVT, then continues through NVE. Early finishers wait for the rest of their group, and the next group begins only when the current group completes both stages.

Start by giving each trajectory its own 50 K velocity draw.

In [5]:
initial_temperature = 50.0  # K for each independent velocity draw
random_seed = 42  # reproducible velocity initialization

Now load the 205 structures, rebuild the Module 02 interaction model, and prepare the selected number of trajectories on `device`.

In [13]:
trajectories, interaction_model = helpers.prepare_ncia250_trajectories(
    device,
    batch_multiplier=batch_multiplier,
    initial_temperature=initial_temperature,
    random_seed=random_seed)

`trajectories` holds the prepared trajectory records, `interaction_model` is shared by every route, and `device` selects where their tensors live.

### One pipeline, two MD stages

Every trajectory starts from its own 50 K velocity draw. We will build NVT first, then continue the same positions and velocities with NVE. Each stage gets its timestep, time limit, and stopping rule. 

`status = 0` runs NVT, `status = 1` runs NVE, and `status = 2` marks completion. The rolling rules let different trajectories change status at different times, giving the scheduling routes useful work to manage.

We will run the same NVT → NVE study three ways:

1. **Run in stages:** one group completes NVT before any trajectory in that group begins NVE. The next group waits until the current group completes both stages.
2. **Fused stages:** within one group, each trajectory begins NVE as soon as its own NVT stopping rule passes. The group keeps the same trajectories until all of them complete.
3. **Fused stages with inflight batching:** trajectories still change stages independently. As completed trajectories leave, waiting trajectories enter the active `Batch`.

Each stopping rule remembers recent values for every trajectory. Those histories belong to one run, so every route builds fresh NVT and NVE stages.

Both rules sample every 10 MD steps. With a 1 fs timestep, 10 samples cover 100 fs. Two passing windows move a trajectory to the next status.

In [7]:
sample_every = 10  # MD steps between screening samples
window_samples = 10  # 10 samples × 10 steps = 100 fs
required_windows = 2  # passing windows before a status change

Different trajectories heat at different rates. Every 10 fs, the NVT rule records the current temperature. A 100 fs window passes when its mean is within 60 K of 300 K and its recent temperature change is within 45 K. Two consecutive passing windows move that trajectory to NVE.

These broad thresholds create different stopping times for this scheduling exercise; they are not a production equilibration protocol. `temperature_window_hook()` stores each rolling history on `device` and changes status `0 → 1`.

In [ ]:
def build_temperature_hook():
    # Give each route independent 100 fs temperature histories.
    return helpers.temperature_window_hook(
        total_trajectories=len(trajectories),
        device=device,
        frequency=sample_every,
        window_samples=window_samples,
        required_streak=required_windows,
        target_temperature=300.0,  # K
        temperature_tolerance=60.0,  # K around the target
        trend_tolerance=45.0,  # K between half-window means
    )

Build NVT with a 1 fs timestep, a 300 K target, 0.01 fs⁻¹ Langevin friction, and a 600 fs time limit. The temperature hook decides when each trajectory advances. `keep_temperature_hook_in_control()` reserves status `0 → 1` for that rule, while the model's neighbor hooks update molecular neighbors before force calculations.

In [9]:
from nvalchemi.dynamics import NVE, NVTLangevin


def build_nvt_stage():
    return NVTLangevin(
        model=interaction_model,
        dt=1.0,  # fs per Langevin step
        temperature=300.0,  # K
        friction=0.01,  # fs⁻¹ Langevin friction
        random_seed=random_seed,  # reproducible Langevin noise
        n_steps=600,  # 600 steps = 600 fs
        # Let the rolling temperature hook control status 0 → 1.
        convergence_hook=helpers.keep_temperature_hook_in_control(),
        hooks=[
            # Refresh neighbors before each standalone force evaluation.
            *interaction_model.make_neighbor_hooks(),
            # Store 100 fs temperature windows; change status 0 → 1.
            build_temperature_hook()])

NVE removes the thermostat and continues from the same positions and velocities with 1 fs steps. Total energy should remain approximately steady. Every 10 fs, the rule records total energy per atom; a 100 fs window passes when its span is at most 0.00075 eV per atom.

Two passing windows mark the trajectory complete with status `2`. The stage has a 500 fs time limit.

In [ ]:
def build_energy_hook():
    # Give each route independent 100 fs energy histories.
    return helpers.energy_window_hook(
        total_trajectories=len(trajectories),
        device=device,
        frequency=sample_every,
        window_samples=window_samples,
        required_streak=required_windows,
        span_per_atom=0.00075,  # eV/atom across the window
    )

Build the NVE stage with the same interaction model. `exit_status=2` identifies completed trajectories, the energy hook decides when they reach that status, and the model's hooks update neighbors before force calculations.

In [ ]:
def build_nve_stage():
    return NVE(
        model=interaction_model,
        dt=1.0,  # fs per NVE step
        n_steps=500,  # 500 steps = 500 fs
        exit_status=2,  # status 2 marks a completed trajectory
        hooks=[
            # Refresh neighbors before each standalone force evaluation.
            *interaction_model.make_neighbor_hooks(),
            # Store 100 fs energy windows; change status 1 → 2.
            build_energy_hook()])

The rolling histories live inside the stage objects. `build_md_stages()` creates a fresh NVT and NVE pair whenever a new group or route starts.

In [12]:
def build_md_stages():
    # Return fresh NVT and NVE stages for one group or inflight route.
    return build_nvt_stage(), build_nve_stage()

Dynamics updates positions, velocities, and status in place. Each route therefore starts from its own clone of the prepared trajectories, keeping their starting state identical. `fresh_batch()` packs those clones with Toolkit's public `Batch.from_data_list()`.

In [ ]:
from nvalchemi.data import Batch


def fresh_batch():
    # Clone every prepared trajectory before packing the route's `Batch`.
    return Batch.from_data_list(
        [record.clone() for record in trajectories],
        device=device)

Create Route 1's `Batch`. Its printed trajectory count should match the selected study size, and its tensors should be on `device`.

In [ ]:
sequential_batch = fresh_batch()
print("trajectories in Route 1:", sequential_batch.num_graphs)
print("device:", sequential_batch.device)

### Choose the active `Batch` capacity

All prepared trajectories stay on `device`, while only one active `Batch` enters each model calculation. The molecule limit controls how many trajectories run together; the atom limit accounts for their different sizes. We will allow 64 molecules, multiply by the study's average atoms per molecule, and reuse both limits for every route. These limits define this scheduling exercise; production limits depend on the model and GPU.

In [ ]:
max_batch_size = 64  # molecules allowed in one active Batch
average_atoms = sequential_batch.num_nodes_per_graph.float().mean()
max_atoms = int(torch.ceil(average_atoms * max_batch_size).item())

print(f"average atoms per molecule: {average_atoms.item():.2f}")
print(f"active Batch limits: {max_batch_size} molecules, {max_atoms} atoms")

Split the study into groups that fit the active `Batch` limits. A group closes when adding the next molecule would exceed the molecule or atom limit. Inspect both totals to see which limit shaped each group.

In [ ]:
group_indices = helpers.group_indices_by_capacity(
    sequential_batch,
    max_batch_size=max_batch_size,
    max_atoms=max_atoms)
[
    {
        "molecules": len(indices),
        "atoms": int(sequential_batch.num_nodes_per_graph[indices].sum()),
    }
    for indices in group_indices
]

Create one `Batch` for each group with `Batch.index_select()`. Route 1 will run them one at a time.

In [ ]:
sequential_groups = [
    sequential_batch.index_select(indices) for indices in group_indices
]
print("molecules per Route 1 group:", [group.num_graphs for group in sequential_groups])

Route 1 runs one MD stage across the whole group. The loop stops when every trajectory passes its rule or the stage reaches its time limit. At that limit, any remaining trajectories move on, and we record how each trajectory finished.

In [ ]:
def finish_sequential_stage(group, stage, target_status):
    # Each public `step()` updates this same GPU `Batch`.
    for _ in range(stage.n_steps):
        group, _ = stage.step(group)
        if bool((group.status >= target_status).all()):
            break

    passed_rule = int((group.status >= target_status).sum().item())
    advanced_at_limit = group.num_graphs - passed_rule

    # Move remaining trajectories on when the stage reaches its time limit.
    group.status.clamp_min_(target_status)
    return group, passed_rule, advanced_at_limit

Run every group through NVT and then NVE. Fresh stages keep the rolling histories independent, four counters retain how each stage ended, and `time_route()` measures the complete sequential route with device synchronization.

In [ ]:
sequential_results = []
nvt_passed = nvt_at_limit = 0
nve_passed = nve_at_limit = 0

# Time the complete sequential route with CUDA synchronization.
with helpers.time_route(device) as sequential_run:
    for group in sequential_groups:
        # Give each group independent stages and rolling histories.
        nvt, nve = build_md_stages()
        group, passed, at_limit = finish_sequential_stage(group, nvt, 1)
        nvt_passed += passed
        nvt_at_limit += at_limit
        group, passed, at_limit = finish_sequential_stage(group, nve, 2)
        nve_passed += passed
        nve_at_limit += at_limit
        sequential_results.append(group)

Inspect how each stopping rule ended. A trajectory either passed the rolling rule during the stage or moved on when the stage reached its time limit.

In [ ]:
print("NVT passed rolling rule:", nvt_passed)
print("NVT advanced at 600 fs time limit:", nvt_at_limit)
print("NVE passed rolling rule:", nve_passed)
print("NVE advanced at 500 fs time limit:", nve_at_limit)

Confirm that Route 1 completed every trajectory exactly once on the selected device. The summary reports synchronized time, throughput, and memory. Its replacement count is `0` because Route 1 never refills a running group.

In [ ]:
sequential_result = helpers.summarize_sequential(
    sequential_results,
    sequential_run,
    expected_trajectories=len(trajectories),
    device=device,
)
sequential_result.summary

> **What to check:** `completed trajectories` equals `len(trajectories)`. The rule and time-limit counts show how many trajectories changed stage naturally. The sequential route creates two `Batch`-wide waiting points, so early finishers keep their slots until the whole active `Batch` crosses each boundary.

## Route 2 · Fused stages

<img src="assets/course-roadmap-module03-fused.svg" alt="ALCHEMI Core roadmap with Fuse stages highlighted in Module 03" style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0.75rem 0 0.5rem;">

`FusedStage` keeps both MD stages in one active `Batch`. At each step, status-`0` trajectories receive the NVT update and status-`1` trajectories receive the NVE update. One `Batch` can therefore contain trajectories in both stages.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `nvt + nve`
>
> **Input:** two dynamics stages that share one model  
> **Returns:** a `FusedStage` with NVT at status `0` and NVE at status `1`

In [ ]:
from nvalchemi.dynamics import FusedStage

`nvt + nve` creates the `FusedStage`. Both stages use the same interaction model. The fused route evaluates that model for the active `Batch`, then applies the NVT or NVE update selected by each trajectory's status. Register the model's neighbor hooks on the fused route so every force calculation uses current neighbors. `build_fused_route()` returns one fresh fused route for each group.

In [ ]:
def build_fused_route():
    # Give each group fresh stages and rolling histories.
    nvt, nve = build_md_stages()
    fused = nvt + nve  # status 0 = NVT; status 1 = NVE

    # Refresh neighbors before each shared fused model evaluation.
    for hook in interaction_model.make_neighbor_hooks():
        fused.register_hook(hook)

    return fused

Create Route 2 from the same prepared positions and velocities. Confirm its trajectory count and device before placing the same molecules together as in Route 1.

In [ ]:
fused_batch = fresh_batch()
print("trajectories in Route 2:", fused_batch.num_graphs)
print("device:", fused_batch.device)

Use Route 1's `group_indices` so both routes process the same molecules together. `Batch.index_select()` packs each group for the fused route; confirm the molecule counts match Route 1.

In [ ]:
fused_groups = [fused_batch.index_select(indices) for indices in group_indices]
print("molecules per fused group:", [group.num_graphs for group in fused_groups])

Build one `FusedStage` for each group. This keeps the stopping-rule histories separate, just as Route 1 did.

In [ ]:
# Give each group independent stages and rolling histories.
fused_routes = [build_fused_route() for _ in fused_groups]
len(fused_routes)

Run each group with its paired `FusedStage`. `run()` lets trajectories change between NVT and NVE by status until the group completes, while the synchronized timer covers the full route.

In [ ]:
fused_results = []

# Time the complete FusedStage route with CUDA synchronization.
with helpers.time_route(device) as fused_run:
    for group, fused in zip(fused_groups, fused_routes, strict=True):
        # `run()` updates and returns this completed GPU `Batch`.
        fused_results.append(fused.run(group))

Confirm that Route 2 completed the same trajectory identities on the same device, then report the same measurements as Route 1.

In [ ]:
# Check the completed Batches against the expected identities and timer.
fused_result = helpers.summarize_fused(
    fused_results,
    fused_run,
    expected_trajectories=len(trajectories),
    device=device,
)

# Display the checked route measurements.
fused_result.summary

Record one representative fused group outside the timed comparison so stage mixing is visible. Start from a fresh `Batch` and confirm its trajectory count and device.

In [ ]:
trace_batch = fresh_batch()
print("trajectories available for the trace:", trace_batch.num_graphs)
print("device:", trace_batch.device)

Select the first Route 1 group for the trace. Using the same molecules and capacity makes the trace representative of the fused route above.

In [ ]:
trace_group = trace_batch.index_select(group_indices[0])
print("molecules in the trace group:", trace_group.num_graphs)

Attach a lightweight recorder that counts NVT and NVE trajectories every 10 fused steps. Run the group once to capture how work splits between the two stages.

In [ ]:
# Build one fresh route and one GPU stage-count recorder.
trace_route = build_fused_route()
stage_trace = helpers.stage_trace_hook(frequency=sample_every)

# Register the recorder, then run the unmodified group through NVT → NVE.
trace_route.register_hook(stage_trace)
trace_group = trace_route.run(trace_group)

Show the first snapshots containing both NVT and NVE work. A row with two nonzero counts confirms that trajectories changed stages independently inside the same `Batch`.

In [ ]:
helpers.summarize_stage_trace(stage_trace)

> **What to check:** at least one row contains both NVT and NVE trajectories. `FusedStage` removes the `Batch`-wide stage boundary. Completed trajectories still occupy slots until the active `Batch` finishes.

## Route 3 · Fused stages with inflight batching

<img src="assets/course-roadmap-module03-inflight.svg" alt="ALCHEMI Core roadmap with Refill active Batch highlighted in Module 03" style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0.75rem 0 0.5rem;">

Route 3 keeps the per-trajectory NVT to NVE routing from Route 2 and adds waiting, active, and completed work. When a completed trajectory leaves the active `Batch`, the sampler supplies another trajectory within the same capacity limits. All trajectory state stays on CUDA.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `InMemoryDataset(fresh_batch())`
>
> **Input:** a new GPU-resident `Batch` of prepared molecular trajectories  
> **Returns:** waiting trajectories held in that in-memory `Batch` on the same device

Confirm the waiting count and device before choosing what enters the active `Batch`.

In [ ]:
from nvalchemi.data.datapipes import InMemoryDataset

# Keep the GPU-resident trajectories in place for indexed sampling.
waiting = InMemoryDataset(fresh_batch())
print("waiting trajectories:", len(waiting))
print("waiting device:", waiting.in_memory_batch.device)

The waiting dataset holds trajectories that have not entered the active `Batch`. `SizeAwareSampler` fills that `Batch` without crossing the molecule or atom limit, then uses newly released space for replacements.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `SizeAwareSampler(waiting, max_batch_size=max_batch_size, max_atoms=max_atoms)`
>
> **Input:** the waiting dataset and the capacity calculated in Route 1  
> **Effect:** fills the first active `Batch` and supplies waiting trajectories as space opens

Route 3 uses the same limits as Routes 1 and 2, so the scheduling comparison keeps one active capacity.

In [ ]:
from nvalchemi.dynamics import SizeAwareSampler

# Apply the same capacity used by the first two routes.
sampler = SizeAwareSampler(
    waiting,
    max_batch_size=max_batch_size,
    max_atoms=max_atoms,
)
print("sampler molecule limit:", sampler.max_batch_size)
print("sampler atom limit:", sampler.max_atoms)

When a trajectory reaches status `2`, `FusedStage` moves it out of the active `Batch` and writes its final state to a Toolkit `DataSink`. This handoff frees space for the sampler to add another waiting trajectory and keeps the completed result available after the active `Batch` changes.

Here we use `helpers.CudaCompletionSink`, a small tutorial implementation of `DataSink`. It stores each trajectory's final positions, velocities, `simulation_id`, and `status` on `device`. Setting its capacity to `len(trajectories)` gives it room for every completed result.

In [ ]:
# Build the tutorial DataSink that keeps completed trajectories on device.
completed = helpers.CudaCompletionSink(
    capacity=len(trajectories),
    device=device,
)

print("completion capacity:", completed.capacity)
print("device:", completed.device)

Route 2 needed only the two MD stages, so `nvt + nve` was enough. Inflight batching also needs the waiting-work sampler, the completion sink, and a refill cadence. The full `FusedStage(...)` constructor connects all four pieces.

One fused update advances every active trajectory by one MD step in its current stage. `refill_frequency=5` checks for open space after every five such updates, so completed work can wait up to five updates for the next refill check. `inflight_mode` confirms that the sampler enabled refills.

In [ ]:
# Create fresh stages and histories for the inflight route.
nvt, nve = build_md_stages()

inflight = FusedStage(
    sub_stages=[(0, nvt), (1, nve)],  # status 0 = NVT, 1 = NVE
    sampler=sampler,  # fills open spaces with waiting trajectories
    sinks=[completed],  # collects completed trajectories on device
    refill_frequency=5,  # fused updates between refill checks
    # Refresh neighbors before each shared fused model evaluation.
    hooks=interaction_model.make_neighbor_hooks(),
)
print("inflight mode:", inflight.inflight_mode)

`inflight.run()` asks the sampler to fill the first active `Batch`. After every five updates, completed trajectories move to the sink and waiting trajectories enter any open molecule and atom capacity.

The call returns `None` when every waiting and active trajectory has completed.

In [ ]:
# Time the route from the first active Batch to final completion.
with helpers.time_route(device) as inflight_run:
    # `None` means every waiting and active trajectory completed.
    remaining = inflight.run()

In [ ]:
# Read the sink and verify that every trajectory completed once on device.
inflight_result = helpers.check_inflight_run(
    completed,
    remaining,
    inflight_run,
    expected_trajectories=len(trajectories),
    device=device,
    max_batch_size=max_batch_size,
)

# Show the same route measurements used above.
inflight_result.summary

Read the completion sink and confirm that every trajectory identity appears exactly once on `device`. The summary also reports how many trajectories entered after the first active `Batch`; a study that fits at once reports `0` replacements.

## Compare the three routes

<img src="assets/course-roadmap-module03-compare.svg" alt="ALCHEMI Core roadmap with Compare routes highlighted in Module 03" style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0.75rem 0 0.5rem;">

Place the three results side by side. Every row covers the same trajectories, model, initial state, MD settings, stopping rules, and active capacity. Start with `completed trajectories` to confirm equal study scope.

Each timer includes the route exactly as called, including its own startup work. Treat this table as a live scheduling walkthrough. A speedup claim needs repeated runs with identical timing boundaries.

In [ ]:
helpers.compare_routes(sequential_result, fused_result, inflight_result)

> **Bottom line**
>
> `FusedStage` lets each trajectory move from NVT to NVE as soon as its stopping rule passes. Inflight batching fills the released space with waiting trajectories. Together they reduce waiting inside the active `Batch` and keep it supplied.

Docs: [Multi-stage pipeline example](https://nvidia.github.io/nvalchemi-toolkit/examples/intermediate/01_multistage_pipeline.html) · [Inflight batching example](https://nvidia.github.io/nvalchemi-toolkit/examples/intermediate/04_inflight_batching.html)

Read `completed trajectories` first to confirm equal study scope. Then compare synchronized time and throughput. `replacement trajectories` reports how many trajectories entered after the first active `Batch`. With the default 410 trajectories, a nonzero value confirms that inflight refills occurred.


## Continue to multiple GPUs

This module used one GPU for many independent trajectories. Toolkit also supports three multi-GPU layouts:

- A distributed pipeline assigns workflow stages to GPU ranks and streams molecular trajectories between them.
- Domain decomposition divides one very large molecule or material across spatial regions owned by several ranks.
- A two-dimensional pipeline combines stage-specific GPU groups with domain decomposition inside each stage.

<img src="assets/03-multi-gpu-continuations-v1.svg" alt="The Module 03 one-GPU study followed by three Toolkit multi-GPU layouts: DistributedPipeline streams trajectories through stage-specific GPU ranks, DomainParallel divides one large system among spatial domains, and pipeline by domain assigns a group of domain ranks to each stage." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0.8rem 0 0.55rem;">

Examples: [Distributed FIRE → Langevin pipeline](https://nvidia.github.io/nvalchemi-toolkit/examples/distributed/01_distributed_pipeline.html) · [Domain-decomposed MACE NVT](https://nvidia.github.io/nvalchemi-toolkit/examples/distributed/03_mace_nvt_distributed.html) · [2-D pipeline × domain decomposition](https://nvidia.github.io/nvalchemi-toolkit/examples/distributed/07_fire_nvt_dd.html)


You have completed ALCHEMI Core Tutorial! 🎉

**Next step:** ~~[Return to the course overview →](../../README.md)~~   **CREATE**, **INNOVATE**, **DISCOVER** with Toolkit